In [ ]:

!git clone https://github.com/viveknarayana/ecs174-dior.git /content/ecs174-dior
%cd /content/ecs174-dior

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

fatal: destination path '/content/ecs174-dior' already exists and is not an empty directory.
/content/ecs174-dior
GPU: Tesla T4


In [ ]:
# Download and extract the four public ZIPs. This transfers about 7 GB; run when ready.
!pip -q install gdown
!mkdir -p /content/dior_archives /content/DIOR
!gdown --folder 'https://drive.google.com/drive/folders/15Pw6i5mkmWtZ1brrUaHpqAyizIzwAoSR?usp=sharing' -O /content/dior_archives

from pathlib import Path
from zipfile import ZipFile
archive_dir = Path('/content/dior_archives')
data_root = Path('/content/DIOR')
for archive in sorted(archive_dir.glob('*.zip')):
    print('Extracting', archive.name)
    with ZipFile(archive) as zf:
        zf.extractall(data_root)
print('Extracted:', sorted(p.name for p in data_root.iterdir()))

Retrieving folder contents
Processing file 1jjVgSg6d0e2pTDe0-erIyC25c35yCOCv Annotations.zip
Processing file 1m43lZAv5cCmc5vQSCA8ELJtmsGhp_2ab ImageSets.zip
Processing file 1jfcyA0Cv3MRPkbf72bJUOEfeMCMkNw5d JPEGImages-test.zip
Processing file 1rFIb68L3YOUXoyDNCp6yrU162sjRyJ4o JPEGImages-trainval.zip
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1jjVgSg6d0e2pTDe0-erIyC25c35yCOCv
From (redirected): https://drive.google.com/uc?id=1jjVgSg6d0e2pTDe0-erIyC25c35yCOCv&confirm=t&uuid=0bf91714-b998-42eb-bb26-2de6d8451287
To: /content/dior_archives/Annotations.zip
100% 32.1M/32.1M [00:00<00:00, 166MB/s]
Downloading...
From: https://drive.google.com/uc?id=1m43lZAv5cCmc5vQSCA8ELJtmsGhp_2ab
To: /content/dior_archives/ImageSets.zip
100% 49.1k/49.1k [00:00<00:00, 87.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1jfcyA0Cv3MRPkbf72bJUOEfeMCMkNw5d
From (redirec

In [ ]:
# Pull the latest project updates before using the cloned source.
!git -C /content/ecs174-dior pull origin main

From https://github.com/viveknarayana/ecs174-dior
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
# Train one epoch, save outputs to Drive, and request local downloads.
import csv
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive, files

drive.mount('/content/drive')

OUTPUT_ROOT = Path('/content/drive/MyDrive/DIOR_checkpoints')


def run_one_epoch(backbone='resnet50', resume=None, previous_log=None):
    run_dir = OUTPUT_ROOT / backbone
    run_dir.mkdir(parents=True, exist_ok=True)

    checkpoint = Path(resume) if resume else None
    log_path = run_dir / 'training_log.csv'
    previous_epoch = 0

    if checkpoint:
        if not checkpoint.is_file():
            raise FileNotFoundError(f'Checkpoint not found: {checkpoint}')

        saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
        previous_epoch = int(saved['epoch'])
        del saved

        # Supply previous_log when bringing a matching CSV into this folder.
        source_log = Path(previous_log) if previous_log else log_path
        if not source_log.is_file():
            raise FileNotFoundError(f'Matching training log not found: {source_log}')

        with source_log.open(newline='') as f:
            rows = list(csv.DictReader(f))

        if not rows or int(rows[-1]['epoch']) != previous_epoch:
            raise ValueError('The CSV must end at the checkpoint epoch.')

        if source_log.resolve() != log_path.resolve():
            shutil.copy2(source_log, log_path)

        print(f'Resuming from epoch {previous_epoch}.', flush=True)

    elif list(run_dir.glob('checkpoint_epoch_*.pth')):
        raise ValueError('Existing checkpoints found. Pass one with resume.')

    next_epoch = previous_epoch + 1
    if next_epoch > 10:
        raise ValueError('The 10-epoch target is already complete.')

    command = [
        sys.executable, '-u', 'train.py',
        '--data-root', '/content/DIOR',
        '--backbone', backbone,
        '--output-root', str(OUTPUT_ROOT),
        '--epochs', str(next_epoch),
        '--batch-size', '2',
    ]

    if checkpoint:
        command += ['--resume', str(checkpoint)]

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)

    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, command)

    print(f'Epoch {next_epoch} saved to {run_dir}', flush=True)

    for name in (
        f'checkpoint_epoch_{next_epoch:02d}.pth',
        'training_log.csv',
        'loss_curves.png',
    ):
        path = run_dir / name
        if not path.is_file():
            raise FileNotFoundError(path)
        print(f'Requesting download: {name}', flush=True)
        files.download(str(path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# One epoch on two training images and one validation image.
run_training('resnet50', ['--tiny-test', '--tiny-train-images', '2', '--tiny-validation-images', '1', '--epochs', '1'], output_root='/content/smoke_test_outputs')

In [ ]:
# Resume from epoch 1, train epoch 2, and request the output downloads.
run_one_epoch(
    'resnet50',
    resume='/content/drive/MyDrive/resnet50_trainval_size800_batch2_lr0.005_epochs10_20261117_002901_epoch01_checkpoint_epoch_01.pth'
)

Resuming from epoch 1.
Training split: trainval (11725 images); validation: disabled
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth

  0%|          | 0.00/160M [00:00<?, ?B/s]
  7%|▋         | 11.9M/160M [00:00<00:01, 124MB/s]
 17%|█▋        | 26.5M/160M [00:00<00:00, 141MB/s]
 28%|██▊       | 44.8M/160M [00:00<00:00, 164MB/s]
 39%|███▉      | 62.6M/160M [00:00<00:00, 172MB/s]
 50%|█████     | 80.2M/160M [00:00<00:00, 177MB/s]
 64%|██████▍   | 102M/160M [00:00<00:00, 195MB/s] 
 76%|███████▌  | 121M/160M [00:00<00:00, 181MB/s]
 87%|████████▋ | 138M/160M [00:00<00:00, 170MB/s]
 98%|█████████▊| 157M/160M [00:00<00:00, 178MB/s]
100%|██████████| 160M/160M [00:00<00:00, 174MB/s]
Resuming at epoch 2
Epoch 2/2 | images 100/11725 | batches 50/5863 | elapsed 0.4 min | average loss 0.3920
Epoch 2/2 | images 200/11725 | batches 100/5863 | elapsed 0.8 min | average loss 0.36

KeyboardInterrupt: 

In [ ]:
# Resume from epoch 2, train epoch 3, and download the outputs.
run_one_epoch(
    'resnet50',
    resume='/content/drive/MyDrive/resnet50_trainval_size800_batch2_lr0.005_epochs10_20261117_002901_epoch02_checkpoint_epoch_02.pth',
    previous_log='/content/dior_outputs/resnet50/training_log.csv',
)

Resuming from epoch 2.
Training split: trainval (11725 images); validation: disabled
Resuming at epoch 3
Epoch 3/3 | images 100/11725 | batches 50/5863 | elapsed 0.4 min | average loss 0.3114
Epoch 3/3 | images 200/11725 | batches 100/5863 | elapsed 0.7 min | average loss 0.2962
Epoch 3/3 | images 300/11725 | batches 150/5863 | elapsed 1.1 min | average loss 0.3224
Epoch 3/3 | images 400/11725 | batches 200/5863 | elapsed 1.5 min | average loss 0.3127
Epoch 3/3 | images 500/11725 | batches 250/5863 | elapsed 1.9 min | average loss 0.3164
Epoch 3/3 | images 600/11725 | batches 300/5863 | elapsed 2.3 min | average loss 0.3191
Epoch 3/3 | images 700/11725 | batches 350/5863 | elapsed 2.6 min | average loss 0.3231
Epoch 3/3 | images 800/11725 | batches 400/5863 | elapsed 3.0 min | average loss 0.3184
Epoch 3/3 | images 900/11725 | batches 450/5863 | elapsed 3.4 min | average loss 0.3211
Epoch 3/3 | images 1000/11725 | batches 500/5863 | elapsed 3.8 min | average loss 0.3207
Epoch 3/3 | ima

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Requesting download: training_log.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Requesting download: loss_curves.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
run_one_epoch(
    'resnet50',
    resume='/content/drive/MyDrive/DIOR_checkpoints/resnet50/checkpoint_epoch_03.pth',
)

Resuming from epoch 3.
Training split: trainval (11725 images); validation: disabled
Resuming at epoch 4
Epoch 4/4 | images 100/11725 | batches 50/5863 | elapsed 0.4 min | average loss 0.3184
Epoch 4/4 | images 200/11725 | batches 100/5863 | elapsed 0.8 min | average loss 0.3088
Epoch 4/4 | images 300/11725 | batches 150/5863 | elapsed 1.2 min | average loss 0.2987
Epoch 4/4 | images 400/11725 | batches 200/5863 | elapsed 1.6 min | average loss 0.3080
Epoch 4/4 | images 500/11725 | batches 250/5863 | elapsed 1.9 min | average loss 0.2960
Epoch 4/4 | images 600/11725 | batches 300/5863 | elapsed 2.3 min | average loss 0.2887
Epoch 4/4 | images 700/11725 | batches 350/5863 | elapsed 2.7 min | average loss 0.2910
Epoch 4/4 | images 800/11725 | batches 400/5863 | elapsed 3.1 min | average loss 0.2878
Epoch 4/4 | images 900/11725 | batches 450/5863 | elapsed 3.5 min | average loss 0.2943
Epoch 4/4 | images 1000/11725 | batches 500/5863 | elapsed 3.9 min | average loss 0.2947
Epoch 4/4 | ima

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Requesting download: training_log.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Requesting download: loss_curves.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.flush_and_unmount()

In [ ]:
# Full MobileNetV3 run.
run_training('mobilenet')

In [ ]:
# Load the smoke-test checkpoint and inspect predictions on one training image.
import sys
sys.path.insert(0, '/content/ecs174-dior/DIOR_Data')
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from dataset import DIORDataset
from train import build_model
from classes import CLASS_NAMES

checkpoint_path = '/content/smoke_test_outputs/resnet50/downloads/resnet50_smoke_size640_batch2_lr0.005_epochs1_20261116_231410_epoch01_checkpoint_epoch_01.pth'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model = build_model(checkpoint['backbone'], checkpoint['config']['image_size'])
model.load_state_dict(checkpoint['model_state_dict'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()
dataset = DIORDataset('/content/DIOR', split='train')
image, target = dataset[0]
with torch.inference_mode():
    prediction = model([image.to(device)])[0]
print('Checkpoint epoch:', checkpoint['epoch'], 'Image ID:', dataset.ids[0])
print('Top 5 predicted classes and confidence scores:')
for score, label in zip(prediction['scores'][:5].cpu(), prediction['labels'][:5].cpu()):
    print(f'{CLASS_NAMES[label.item()-1]}: {score.item():.3f}')
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image.permute(1, 2, 0))
for box, score, label in zip(prediction['boxes'].cpu(), prediction['scores'].cpu(), prediction['labels'].cpu()):
    if score < 0.25:
        continue
    x1, y1, x2, y2 = box.tolist()
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='red', linewidth=2))
    ax.text(x1, y1, f'{CLASS_NAMES[label.item()-1]} {score:.2f}', color='white', backgroundcolor='red')
ax.set_title('Smoke-test predictions, confidence ≥ 0.25')
ax.axis('off')
plt.show()

In [ ]:
from pathlib import Path

for p in Path('/content/drive').rglob('*.pth'):
    print(p)

/content/drive/MyDrive/resnet50_trainval_size800_batch2_lr0.005_epochs10_20261117_002901_epoch01_checkpoint_epoch_01.pth
